# 📚 Bookstore Web Scraping Pipeline — Step by Step
**Author:** Oluwasegun Raphael  
**Tools:** Python, Requests, BeautifulSoup, Pandas  

---
### What this notebook does
I scrape **books.toscrape.com** step by step,  from fetching a single page all the way to a clean, exported dataset.
Run each cell one at a time and read the comments to understand what's happening.


---
## Import Libraries
These are all the tools i used. Each one has a specific job.

In [12]:
import requests                          # fetches web pages (like a browser)
from bs4 import BeautifulSoup            # reads and parses the HTML we fetch
import pandas as pd                      # stores and cleans our data
import time                              # adds delays so we don't overload the server
import os                                # handles file paths and folders
from datetime import datetime            # timestamps our output files

print('✅ All libraries imported successfully')

✅ All libraries imported successfully


---
## Configuration
Here i set up the base URL and headers. Headers make scraper look like a real browser so the website doesn't block when scrapping.

In [2]:
# The website i'm scraping
BASE_URL = "https://books.toscrape.com"

# This tells the website we are a normal browser, not a bot
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# Converts word ratings to numbers
RATING_MAP = {
    "One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5
}

print(f'✅ Target website: {BASE_URL}')
print(f'✅ Rating map: {RATING_MAP}')

✅ Target website: https://books.toscrape.com
✅ Rating map: {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}


---
## Fetch a Single Page
Before scraping everything, let's just fetch the homepage and see what we get.

In [3]:
# Send a GET request to the homepage
response = requests.get(BASE_URL, headers=HEADERS, timeout=10)

# Check if it worked — 200 means success
print(f'Status code: {response.status_code}')
print(f'Page size  : {len(response.text)} characters')

# Show the first 500 characters of raw HTML
print('\n--- First 500 characters of raw HTML ---')
print(response.text[:500])

Status code: 200
Page size  : 51294 characters

--- First 500 characters of raw HTML ---
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


---
## Parse the HTML with BeautifulSoup
Raw HTML is messy. I use BeautifulSoup turns it into something i can easily navigate and search.

In [4]:
# Parse the raw HTML
soup = BeautifulSoup(response.text, "lxml")

# Get the page title
print(f'Page title: {soup.title.text}')

# Count how many book cards are on the homepage
books_on_page = soup.select('article.product_pod')
print(f'Books on homepage: {len(books_on_page)}')

# Show raw HTML of the first book card
print('\n--- First book card HTML ---')
print(books_on_page[0].prettify())

Page title: 
    All products | Books to Scrape - Sandbox

Books on homepage: 20

--- First book card HTML ---
<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</articl

---
## Extract Data from One Book
Now that i understand what the HTML looks like, let me pull out the fields i actually want from a single book.

In [5]:
# Take the first book card
book = books_on_page[0]

# Extract title
title = book.h3.a["title"]

# Extract price and clean it up
price_raw = book.select_one("p.price_color").text.strip()
price = float(price_raw.replace("Â", "").replace("£", "").strip())

# Extract rating (it's stored as a word in the CSS class)
rating_word = book.p["class"][1]       # e.g. 'Three'
rating = RATING_MAP.get(rating_word, 0) # convert to number

# Extract availability
availability = book.select_one("p.availability").text.strip()

# Extract the book's URL
relative_url = book.h3.a["href"].replace("../", "")
full_url = f"{BASE_URL}/catalogue/{relative_url}"

print(f'Title       : {title}')
print(f'Price       : £{price}')
print(f'Rating      : {rating_word} -> {rating} stars')
print(f'Availability: {availability}')
print(f'URL         : {full_url}')

Title       : A Light in the Attic
Price       : £51.77
Rating      : Three -> 3 stars
Availability: In stock
URL         : https://books.toscrape.com/catalogue/catalogue/a-light-in-the-attic_1000/index.html


---
## Extract All Books from the Homepage
With this i acn now loop through all 20 books on the homepage and store them in a list.

In [6]:
all_books = []

for book in books_on_page:
    title        = book.h3.a["title"]
    price_raw    = book.select_one("p.price_color").text.strip()
    price        = float(price_raw.replace("Â", "").replace("£", "").strip())
    rating_word  = book.p["class"][1]
    rating       = RATING_MAP.get(rating_word, 0)
    availability = book.select_one("p.availability").text.strip()
    relative_url = book.h3.a["href"].replace("../", "")
    full_url     = f"{BASE_URL}/catalogue/{relative_url}"

    all_books.append({
        "title"       : title,
        "price_gbp"   : price,
        "rating"      : rating,
        "availability": availability,
        "url"         : full_url
    })

# Preview as a DataFrame
df_preview = pd.DataFrame(all_books)
print(f'Books extracted: {len(df_preview)}')
df_preview.head()

Books extracted: 20


,title,price_gbp,rating,availability,url
0,A Light in the Attic,51.77,3,In stock,https://books.toscrape.com/catalogue/catalogue...
1,Tipping the Velvet,53.74,1,In stock,https://books.toscrape.com/catalogue/catalogue...
2,Soumission,50.10,1,In stock,https://books.toscrape.com/catalogue/catalogue...
3,Sharp Objects,47.82,4,In stock,https://books.toscrape.com/catalogue/catalogue...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,https://books.toscrape.com/catalogue/catalogue...


---
## Handle Pagination
The homepage only shows 20 books. Most categories have multiple pages. Let's see how pagination works.

In [7]:
# Check if there is a 'next' button on the homepage
next_button = soup.select_one("li.next > a")

if next_button:
    print(f'Next page link found: {next_button["href"]}')
else:
    print('No next page — this is the last page')

# Checking pagination on a category with many books
fiction_url = BASE_URL + "/catalogue/category/books/fiction_10/index.html"
fiction_response = requests.get(fiction_url, headers=HEADERS)
fiction_soup = BeautifulSoup(fiction_response.text, "lxml")

next_btn = fiction_soup.select_one("li.next > a")
total_books = fiction_soup.select_one("form strong").text

print(f'\nFiction category — total books: {total_books}')
print(f'Next page href: {next_btn["href"] if next_btn else "None"}')

Next page link found: catalogue/page-2.html

Fiction category — total books: 65
Next page href: page-2.html


---
## Scrape All Pages of One Category
Here i combine extraction + pagination to scrape every book in the Fiction category.

In [8]:
def scrape_category(category_name, category_url):
    """
    Scrapes all pages of a single category.
    Keeps following the 'next' button until there are no more pages.
    """
    books    = []
    page_url = category_url
    page_num = 1

    while page_url:
        print(f'  Scraping page {page_num}: {page_url}')

        response = requests.get(page_url, headers=HEADERS, timeout=10)
        soup     = BeautifulSoup(response.text, "lxml")

        # Extract all books on this page
        for book in soup.select("article.product_pod"):
            title        = book.h3.a["title"]
            price_raw    = book.select_one("p.price_color").text.strip()
            price        = float(price_raw.replace("Â", "").replace("£", "").strip())
            rating_word  = book.p["class"][1]
            rating       = RATING_MAP.get(rating_word, 0)
            availability = book.select_one("p.availability").text.strip()
            relative_url = book.h3.a["href"].replace("../", "")

            books.append({
                "title"       : title,
                "category"    : category_name,
                "price_gbp"   : price,
                "rating"      : rating,
                "availability": availability,
                "url"         : f"{BASE_URL}/catalogue/{relative_url}"
            })

        # Check for next page
        next_btn = soup.select_one("li.next > a")
        if next_btn:
            base     = page_url.rsplit("/", 1)[0]
            page_url = base + "/" + next_btn["href"]
            page_num += 1
            time.sleep(0.5)   # be polite — don't hammer the server
        else:
            page_url = None   # no more pages, stop the loop

    print(f'  Done — {len(books)} books scraped from {category_name}')
    return books


# Test it on Fiction category
fiction_books = scrape_category(
    "Fiction",
    BASE_URL + "/catalogue/category/books/fiction_10/index.html"
)

pd.DataFrame(fiction_books).head(10)

  Scraping page 1: https://books.toscrape.com/catalogue/category/books/fiction_10/index.html
  Scraping page 2: https://books.toscrape.com/catalogue/category/books/fiction_10/page-2.html
  Scraping page 3: https://books.toscrape.com/catalogue/category/books/fiction_10/page-3.html
  Scraping page 4: https://books.toscrape.com/catalogue/category/books/fiction_10/page-4.html
  Done — 65 books scraped from Fiction


,title,category,price_gbp,rating,availability,url
0,Soumission,Fiction,50.10,1,In stock,https://books.toscrape.com/catalogue/soumissio...
1,Private Paris (Private #10),Fiction,47.61,5,In stock,https://books.toscrape.com/catalogue/private-p...
2,"We Love You, Charlie Freeman",Fiction,50.27,5,In stock,https://books.toscrape.com/catalogue/we-love-y...
3,Thirst,Fiction,17.27,5,In stock,https://books.toscrape.com/catalogue/thirst_94...
4,The Murder That Never Was (Forensic Instincts #5),Fiction,54.11,3,In stock,https://books.toscrape.com/catalogue/the-murde...
5,Tuesday Nights in 1980,Fiction,21.04,2,In stock,https://books.toscrape.com/catalogue/tuesday-n...
6,The Vacationers,Fiction,42.15,4,In stock,https://books.toscrape.com/catalogue/the-vacat...
7,The Regional Office Is Under Attack!,Fiction,51.36,5,In stock,https://books.toscrape.com/catalogue/the-regio...
8,Finders Keepers (Bill Hodges Trilogy #2),Fiction,53.53,5,In stock,https://books.toscrape.com/catalogue/finders-k...
9,The Time Keeper,Fiction,27.88,5,In stock,https://books.toscrape.com/catalogue/the-time-...


---
## Get All 50 Category URLs
Here i extract each category URL from the sidebar.

In [10]:
# Re-fetch homepage
homepage = requests.get(BASE_URL, headers=HEADERS)
homepage_soup = BeautifulSoup(homepage.text, "lxml")

# Extract all category links from the sidebar
categories = {}
for tag in homepage_soup.select("ul.nav-list > li > ul > li > a"):
    name = tag.text.strip()
    url  = BASE_URL + "/" + tag["href"]
    categories[name] = url

print(f'Total categories found: {len(categories)}')
print('\nFirst 10 categories:')
for name, url in list(categories.items())[:10]:
    print(f'  {name:25} -> {url}')

Total categories found: 50

First 10 categories:
  Travel                    -> https://books.toscrape.com/catalogue/category/books/travel_2/index.html
  Mystery                   -> https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
  Historical Fiction        -> https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
  Sequential Art            -> https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
  Classics                  -> https://books.toscrape.com/catalogue/category/books/classics_6/index.html
  Philosophy                -> https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html
  Romance                   -> https://books.toscrape.com/catalogue/category/books/romance_8/index.html
  Womens Fiction            -> https://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html
  Fiction                   -> https://books.toscrape.com/catalogue/category/books/fiction_

---
## Scrape ALL 50 Categories
Now i put it all together. This scrapes every book across all categories.
If youre running this with me note that this takes about 3-5 minutes — watch the progress.

In [11]:
all_books = []

for i, (name, url) in enumerate(categories.items(), 1):
    print(f'[{i}/50] Scraping: {name}')
    books = scrape_category(name, url)
    all_books.extend(books)
    time.sleep(0.8)   # polite delay between categories

print(f'\nTotal books collected: {len(all_books)}')

[1/50] Scraping: Travel
  Scraping page 1: https://books.toscrape.com/catalogue/category/books/travel_2/index.html
  Done — 11 books scraped from Travel
[2/50] Scraping: Mystery
  Scraping page 1: https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
  Scraping page 2: https://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html
  Done — 32 books scraped from Mystery
[3/50] Scraping: Historical Fiction
  Scraping page 1: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
  Scraping page 2: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/page-2.html
  Done — 26 books scraped from Historical Fiction
[4/50] Scraping: Sequential Art
  Scraping page 1: https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
  Scraping page 2: https://books.toscrape.com/catalogue/category/books/sequential-art_5/page-2.html
  Scraping page 3: https://books.toscrape.com/catalogue/category/books/seq

---
## Build the DataFrame
Converting our list of dictionaries into a Pandas DataFrame.

In [13]:
df = pd.DataFrame(all_books)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nData types:')
print(df.dtypes)

df.head(10)

Shape: (1000, 6)
Columns: ['title', 'category', 'price_gbp', 'rating', 'availability', 'url']

Data types:
title               str
category            str
price_gbp       float64
rating            int64
availability        str
url                 str
dtype: object


,title,category,price_gbp,rating,availability,url
0,It's Only the Himalayas,Travel,45.17,2,In stock,https://books.toscrape.com/catalogue/its-only-...
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Travel,49.43,4,In stock,https://books.toscrape.com/catalogue/full-moon...
2,See America: A Celebration of Our National Par...,Travel,48.87,3,In stock,https://books.toscrape.com/catalogue/see-ameri...
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,2,In stock,https://books.toscrape.com/catalogue/vagabondi...
4,Under the Tuscan Sun,Travel,37.33,3,In stock,https://books.toscrape.com/catalogue/under-the...
5,A Summer In Europe,Travel,44.34,2,In stock,https://books.toscrape.com/catalogue/a-summer-...
6,The Great Railway Bazaar,Travel,30.54,1,In stock,https://books.toscrape.com/catalogue/the-great...
7,A Year in Provence (Provence #1),Travel,56.88,4,In stock,https://books.toscrape.com/catalogue/a-year-in...
8,The Road to Little Dribbling: Adventures of an...,Travel,23.21,1,In stock,https://books.toscrape.com/catalogue/the-road-...
9,Neither Here nor There: Travels in Europe,Travel,38.95,3,In stock,https://books.toscrape.com/catalogue/neither-h...


---
## Data Validation & Cleaning
Check for duplicates, missing values, and data quality issues.

In [14]:
print('=== Before Cleaning ===')
print(f'Total records  : {len(df)}')
print(f'Duplicates     : {df.duplicated(subset=["title", "category"]).sum()}')
print(f'Missing values :\n{df.isnull().sum()}')
print(f'Price <= 0     : {(df["price_gbp"] <= 0).sum()}')

# Remove duplicates
df = df.drop_duplicates(subset=["title", "category"])

# Add a data quality flag column
df["data_quality"] = "OK"
df.loc[df["price_gbp"] <= 0, "data_quality"] = "INVALID_PRICE"
df.loc[df["title"].isnull(), "data_quality"] = "MISSING_TITLE"

print('\n=== After Cleaning ===')
print(f'Total records  : {len(df)}')
print(f'Quality flags  :\n{df["data_quality"].value_counts()}')

=== Before Cleaning ===
Total records  : 1000
Duplicates     : 1
Missing values :
title           0
category        0
price_gbp       0
rating          0
availability    0
url             0
dtype: int64
Price <= 0     : 0

=== After Cleaning ===
Total records  : 999
Quality flags  :
data_quality
OK    999
Name: count, dtype: int64


---
## Summary Statistics
Quick overview of what i scraped.

In [15]:
print('=== Dataset Summary ===')
print(f'Total books      : {len(df)}')
print(f'Categories       : {df["category"].nunique()}')
print(f'Avg price        : £{df["price_gbp"].mean():.2f}')
print(f'Cheapest book    : £{df["price_gbp"].min():.2f}')
print(f'Most expensive   : £{df["price_gbp"].max():.2f}')
print(f'Avg rating       : {df["rating"].mean():.2f} / 5')
print(f'In stock         : {(df["availability"] == "In stock").sum()}')
print(f'Out of stock     : {(df["availability"] != "In stock").sum()}')

print('\n=== Books per Category (Top 10) ===')
print(df["category"].value_counts().head(10))

=== Dataset Summary ===
Total books      : 999
Categories       : 50
Avg price        : £35.07
Cheapest book    : £10.00
Most expensive   : £59.99
Avg rating       : 2.92 / 5
In stock         : 999
Out of stock     : 0

=== Books per Category (Top 10) ===
category
Default           152
Nonfiction        110
Sequential Art     75
Add a comment      67
Fiction            65
Young Adult        54
Fantasy            47
Romance            35
Mystery            32
Food and Drink     30
Name: count, dtype: int64


---
## Export to CSV and JSON
Save the clean dataset in two formats for downstream use.

In [16]:
# Create outputs folder if it doesn't exist
output_dir = os.path.join("..", "outputs")
os.makedirs(output_dir, exist_ok=True)

# Timestamp the filename so each run creates a new file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path  = os.path.join(output_dir, f"books_{timestamp}.csv")
json_path = os.path.join(output_dir, f"books_{timestamp}.json")

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2)

print(f'CSV  saved: {csv_path}')
print(f'JSON saved: {json_path}')
print(f'\nFinal dataset shape: {df.shape}')
df.tail(5)

CSV  saved: ..\outputs\books_20260618_122007.csv
JSON saved: ..\outputs\books_20260618_122007.json

Final dataset shape: (999, 7)


,title,category,price_gbp,rating,availability,url,data_quality
995,Why the Right Went Wrong: Conservatism--From G...,Politics,52.65,4,In stock,https://books.toscrape.com/catalogue/why-the-r...,OK
996,Equal Is Unfair: America's Misguided Fight Aga...,Politics,56.86,1,In stock,https://books.toscrape.com/catalogue/equal-is-...,OK
997,Amid the Chaos,Cultural,36.58,1,In stock,https://books.toscrape.com/catalogue/amid-the-...,OK
998,Dark Notes,Erotica,19.19,5,In stock,https://books.toscrape.com/catalogue/dark-note...,OK
999,The Long Shadow of Small Ghosts: Murder and Me...,Crime,10.97,1,In stock,https://books.toscrape.com/catalogue/the-long-...,OK
